# Synthetic Dataset Generator

This notebook generates synthetic datasets using local LLMs (e.g. TinyLlama, Mistral). It supports CPU and GPU, multiple schema types (E-commerce, Customer Support, QA Pairs), and exports to CSV or JSONL. A Gradio UI lets you pick the model, dataset type, row count, and output format.

In [ ]:
#Install dependencies
%pip install -q transformers accelerate bitsandbytes datasets pandas gradio huggingface_hub

In [ ]:
#imports
import json
import re
import random
import time
import pandas as pd
import torch

import gradio as gr

from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM

try:
    from transformers import BitsAndBytesConfig
    _BITSANDBYTES_AVAILABLE = True
except ImportError:
    BitsAndBytesConfig = None
    _BITSANDBYTES_AVAILABLE = False

In [ ]:
#Schemas
DATASET_TEMPLATES = {

    "E-commerce Products": {
        "schema": ["product_name","category","price_kes","rating","in_stock"],
        "prompt_styles": [
            "Generate realistic Kenyan online store products.",
            "Generate diverse products including premium and budget items.",
            "Generate unusual but plausible product combinations."
        ]
    },

    "Customer Support": {
        "schema": ["ticket_id","customer_issue","priority","agent_reply","resolution_status"],
        "prompt_styles": [
            "Generate customer service tickets for an internet provider.",
            "Generate tickets with angry, calm, and confused customers.",
            "Generate realistic troubleshooting conversations."
        ]
    },

    "QA Pairs": {
        "schema": ["question","answer","topic","difficulty"],
        "prompt_styles": [
            "Generate educational QA pairs.",
            "Generate practical real-world tech questions.",
            "Generate tricky but fair knowledge questions."
        ]
    }
}

In [ ]:
#Strong prompts for guaranteed JSON
def build_row_prompt(dataset_type, schema, style, seed):

    schema_str = ", ".join(schema)

    return f"""


    Return one JSON You are generating synthetic dataset rows.

    DATASET TYPE: {dataset_type}

    SCHEMA (keys must match exactly):
    {schema_str}

    DIVERSITY INSTRUCTION:
    {style}

    RULES:
    - Output ONLY one JSON object
    - Do NOT include explanations
    - Do NOT include markdown
    - Do NOT include lists
    - Keys must match schema exactly

    Example format:

    {{
    {', '.join([f'"{k}": "value"' for k in schema])}
    }}

    Seed: {seed}. Output the JSON object now.
    """

In [ ]:
#JSON extraction (robust: brace-matching, strip markdown, trailing commas)
def extract_json(text):
    if not text or "{" not in text:
        return None
    # Strip markdown code blocks if present
    text = re.sub(r"^.*?```(?:json)?\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"```\s*$", "", text)
    start = text.index("{")
    depth = 0
    for i in range(start, len(text)):
        if text[i] == "{":
            depth += 1
        elif text[i] == "}":
            depth -= 1
            if depth == 0:
                json_text = text[start : i + 1]
                json_text = json_text.replace("\n", " ")
                json_text = re.sub(r",\s*}", "}", json_text)
                try:
                    return json.loads(json_text)
                except Exception:
                    pass
                break
    # Fallback: first {...} greedy match
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if not match:
        return None
    json_text = match.group(0).replace("\n", " ").strip()
    json_text = re.sub(r",\s*}", "}", json_text)
    try:
        return json.loads(json_text)
    except Exception:
        return None

In [ ]:
#critic prompt builder
def build_review_prompt(schema, row):

    schema_str = ", ".join(schema)

    return f"""
    You are a dataset quality reviewer.

    TASK:
    Review the JSON row and fix any issues.

    RULES:
    - Ensure the JSON contains ALL schema fields
    - Ensure values are realistic
    - Ensure the JSON is valid
    - Do NOT remove fields
    - Do NOT add extra fields

    SCHEMA:
    {schema_str}

    ROW:
    {json.dumps(row)}

    Return ONLY the corrected JSON object.
    """

In [ ]:
#Row reviewer
def review_row(pipe, schema, row):

    review_prompt = build_review_prompt(schema, row)

    result = pipe(review_prompt, max_new_tokens=200)

    raw = result[0]["generated_text"]
    # Pipeline often returns prompt + completion; keep only the new part for JSON
    if raw.startswith(review_prompt):
        raw = raw[len(review_prompt):].strip()

    repaired = extract_json(raw)

    if repaired is None:
        return row

    for k in schema:
        repaired.setdefault(k,"")

    return repaired

In [ ]:
#Pipeline loader (CPU or GPU). Uses chat template when the model supports it (e.g. TinyLlama-Chat).
from transformers import GenerationConfig

def load_pipeline_model(model_id):
    device = 0 if torch.cuda.is_available() else -1
    pipe = pipeline(
        "text-generation",
        model=model_id,
        device=device
    )
    tokenizer = pipe.tokenizer
    if hasattr(tokenizer, "apply_chat_template") and getattr(tokenizer, "chat_template", None):
        def wrapped_pipe(prompt, max_new_tokens=200):
            formatted = tokenizer.apply_chat_template(
                [{"role": "user", "content": prompt}],
                tokenize=False,
                add_generation_prompt=True,
            )
            out = pipe(formatted, generation_config=GenerationConfig(max_new_tokens=max_new_tokens))
            raw = out[0]["generated_text"]
            if raw.startswith(formatted):
                raw = raw[len(formatted):].strip()
            return [{"generated_text": raw}]
        return wrapped_pipe
    return pipe

In [ ]:
#Quantized model loader (GPU + bitsandbytes only)
def load_quantized_model(model_id="mistralai/Mistral-7B-Instruct-v0.2"):
    if not _BITSANDBYTES_AVAILABLE:
        raise ImportError("bitsandbytes is required for quantized models. Install with: pip install bitsandbytes")
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
    )
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        device_map="auto",
        quantization_config=bnb_config
    )
    return tokenizer, model

In [ ]:
#Quantized generation
def generate_quantized(tokenizer, model, prompt):

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    output = model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=True,
        temperature=0.9
    )

    text = tokenizer.decode(output[0], skip_special_tokens=True)

    return text

In [ ]:
#Dataset generator
def generate_dataset_rows(model_id, dataset_type, rows, diversity, quantized):

    cfg = DATASET_TEMPLATES[dataset_type]
    schema = cfg["schema"]

    generated_rows = []
    failures = 0

    # Quantized needs GPU and bitsandbytes; on CPU use normal pipeline
    use_quantized = quantized and torch.cuda.is_available() and _BITSANDBYTES_AVAILABLE

    if use_quantized:
        tokenizer, model = load_quantized_model(model_id)
        def pipe(prompt, max_new_tokens=200):
            text = generate_quantized(tokenizer, model, prompt)
            return [{"generated_text": text}]
    else:
        pipe = load_pipeline_model(model_id)

    for i in range(rows):

        style = random.choice(cfg["prompt_styles"]) if diversity == "High" else cfg["prompt_styles"][0]

        prompt = build_row_prompt(dataset_type, schema, style, i)

        if use_quantized:
            raw = generate_quantized(tokenizer, model, prompt)
        else:
            result = pipe(prompt, max_new_tokens=200)
            raw = result[0]["generated_text"]
            if raw.startswith(prompt):
                raw = raw[len(prompt):].strip()

        row = extract_json(raw)

        if row is None:
            failures += 1
            continue

        for key in schema:
            row.setdefault(key,"")

        clean_row = review_row(pipe, schema, row)

        generated_rows.append(clean_row)

    df = pd.DataFrame(generated_rows)
    if df.empty:
        df = pd.DataFrame(columns=schema)

    return df, failures

In [ ]:
#Export + Controller
def run_generation(model_choice, dataset_type, rows, diversity, format_type, quantized):
    schema = DATASET_TEMPLATES.get(dataset_type, {}).get("schema", [])
    empty_df = pd.DataFrame(columns=schema)

    try:
        df, failures = generate_dataset_rows(
            model_choice,
            dataset_type,
            int(rows),
            diversity,
            quantized
        )
    except Exception as e:
        return empty_df, f"Error: {e}", None

    timestamp = int(time.time())
    csv_path = f"synthetic_{timestamp}.csv"
    jsonl_path = f"synthetic_{timestamp}.jsonl"

    df.to_csv(csv_path, index=False)

    with open(jsonl_path, "w") as f:
        for _, row in df.iterrows():
            d = row.to_dict()
            # JSON does not support NaN; use empty string
            d = {k: ("" if pd.isna(v) else v) for k, v in d.items()}
            f.write(json.dumps(d) + "\n")

    file_path = csv_path if format_type == "CSV" else jsonl_path

    status = f"Generated {len(df)} rows | Failed rows: {failures}"
    if not torch.cuda.is_available() and quantized:
        status += " (quantized requires GPU; used CPU pipeline)"

    return df.head(10), status, file_path

In [ ]:
# Quick test (CPU-friendly: 2 rows, TinyLlama). Run this before launching the Gradio UI.
def _test_cpu_generation():
    print("Testing dataset generation (2 rows, TinyLlama)...")
    try:
        df, failures = generate_dataset_rows(
            model_id="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
            dataset_type="E-commerce Products",
            rows=2,
            diversity="Low",
            quantized=False
        )
        print(f"Generated {len(df)} rows, failures: {failures}")
        print(df.head())
        assert len(df) >= 0 and isinstance(df, pd.DataFrame)
        print("Test passed.")
    except Exception as e:
        print("Test failed:", e)
        print("You can still launch the Gradio UI and try generating from the app.")
_test_cpu_generation()

In [ ]:
#Gradio UI
with gr.Blocks() as demo:

    gr.Markdown("# Synthetic Dataset Generator")
    gr.Markdown("On **CPU**, use fewer rows (e.g. 5–10) for faster runs. Leave *Use 4-bit Quantized* unchecked without GPU.")

    with gr.Row():

        model_choice = gr.Dropdown(
            [
                "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
                "mistralai/Mistral-7B-Instruct-v0.2"
            ],
            value="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
            label="Model"
        )

        dataset_type = gr.Dropdown(
            list(DATASET_TEMPLATES.keys()),
            value="E-commerce Products",
            label="Dataset Type"
        )

    with gr.Row():

        rows = gr.Slider(5, 200, value=10, step=1, label="Rows")

        diversity = gr.Dropdown(
            ["Low","High"],
            value="High",
            label="Diversity"
        )

        format_type = gr.Dropdown(
            ["CSV","JSONL"],
            value="CSV",
            label="Output Format"
        )

    quantized = gr.Checkbox(
        label="Use 4-bit Quantized Model (GPU)",
        value=False
    )
    

    generate_button = gr.Button("Generate Dataset")

    preview = gr.Dataframe(label="Preview")
    status = gr.Textbox(label="Status")
    file_download = gr.File(label="Download File")

    generate_button.click(
        run_generation,
        [model_choice,dataset_type,rows,diversity,format_type,quantized],
        [preview,status,file_download]
    )

demo.launch(share=True)